# 08 - Kemas Hasil Akhir

**Tujuan:** Mengumpulkan laporan, model raw, dan model standar HF menjadi satu zip untuk arsip atau dikumpulkan.

| | |
|---|---|
| **Input** | `reports/`, `models/raw/`, `models/hf_export/`, `checkpoints/best.pt` |
| **Output** | `releases/mmfood-results.zip` di Drive |
| **Prasyarat** | Notebook 06 (07 opsional) |

> Setiap notebook berdiri sendiri: ia membaca hasil notebook sebelumnya dari Google Drive, jadi sesi Colab boleh diputus/diulang di antara notebook.

In [4]:
# --- Bootstrap (sama di setiap notebook) ---------------------------------------
# 1) Sambungkan Google Drive (tempat semua hasil disimpan permanen)
import os, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

# 2) Lokasi di Drive: folder ini sekaligus ROOT proyek (zip dan semua hasil ada di sini)
ROOT_ON_DRIVE = "/content/drive/MyDrive/models-train/mmfood-multitask"
ZIP_ON_DRIVE = f"{ROOT_ON_DRIVE}/mmfood-multitask.zip"

# 3) Folder hasil = ROOT di Drive. Harus diset SEBELUM mengimpor config
os.environ["MMFOOD_PROJECT"] = ROOT_ON_DRIVE

# 4) Ekstrak kode + dataset dari zip ke disk lokal Colab (hanya sekali per sesi)
REPO = "/content/mmfood-multitask"
if not os.path.exists(REPO):
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        z.extractall("/content")

# 5) Daftarkan folder src/ agar modul proyek bisa di-import
sys.path.insert(0, f"{REPO}/src")

# 6) Muat path folder (P) dan konfigurasi (CFG) dari src/config.py
from config import get_paths, load_cfg
P = get_paths()       # P.processed, P.checkpoints, P.hf_export, ... (folder dibuat otomatis)
CFG = load_cfg(P)     # nilai bawaan + perubahan dari notebook 00
print("Folder hasil:", P.project)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder hasil: /content/drive/MyDrive/models-train/mmfood-multitask


## Langkah 8.1 - Kumpulkan dan zip
Checkpoint besar `last.pt` (berisi state optimizer) **tidak** ikut; hanya `best.pt`.

In [5]:
import shutil, os

# Folder sementara berisi susunan rapi yang akan di-zip
tmp = "/content/results_pack"
shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)

# Salin bagian-bagian hasil
shutil.copytree(f"{P.project}/reports", f"{tmp}/reports")      # metrik + grafik
shutil.copytree(P.raw_model, f"{tmp}/models/raw")              # model raw
shutil.copytree(P.hf_export, f"{tmp}/models/hf_export")        # model standar HF
shutil.copytree(P.artifacts, f"{tmp}/artifacts")               # config + label map
os.makedirs(f"{tmp}/checkpoints"); shutil.copy(f"{P.checkpoints}/best.pt", f"{tmp}/checkpoints/best.pt")

# Buat zip di Drive
out = shutil.make_archive(f"{P.releases}/mmfood-results", "zip", tmp)
print("Zip hasil:", out, f"({os.path.getsize(out)/1e6:.0f} MB)")

Zip hasil: /content/drive/MyDrive/models-train/mmfood-multitask/releases/mmfood-results.zip (315 MB)


In [6]:
# Daftar isi zip untuk verifikasi
import zipfile
with zipfile.ZipFile(f"{P.releases}/mmfood-results.zip") as z:
    for i in z.infolist():
        print(f"{i.file_size/1e6:8.2f} MB  {i.filename}")

    0.00 MB  artifacts/
    0.00 MB  checkpoints/
    0.00 MB  models/
    0.00 MB  reports/
    0.00 MB  artifacts/preprocess.json
    0.00 MB  artifacts/config.json
    0.00 MB  artifacts/labels.json
    0.00 MB  reports/figures/
    0.00 MB  reports/metrics/
    0.06 MB  reports/figures/confusion_food_type.png
    0.05 MB  reports/figures/eda_label_distribution.png
    1.08 MB  reports/figures/sample_images.png
    0.06 MB  reports/figures/training_curves.png
    0.07 MB  reports/figures/eda_nutrition_distribution.png
    0.00 MB  reports/metrics/history.json
    0.00 MB  reports/metrics/test_metrics.json
    0.00 MB  reports/metrics/eda_stats.json
  112.34 MB  checkpoints/best.pt
    0.00 MB  models/hf_export/
    0.00 MB  models/raw/
    0.00 MB  models/raw/training_config.json
  112.42 MB  models/raw/mmfood_multitask_raw.pt
    0.00 MB  models/raw/labels.json
    0.00 MB  models/hf_export/README.md
  112.30 MB  models/hf_export/model.safetensors
    0.00 MB  models/hf_export/prep